In [14]:
import torch
from torch import nn

# [2, 4]
X = torch.rand(size=(2, 4))

In [15]:
# 동일한 Layer를 재사용한 model

shared = nn.LazyLinear(8)

net = nn.Sequential(
    nn.LazyLinear(8),  # [2, 4] -> [2, 8]
    nn.ReLU(),
    shared,  # [2, 8] -> [2, 8]
    nn.ReLU(),
    shared,  # [2, 8] -> [2, 8]
    nn.ReLU(),
    nn.LazyLinear(1),  # [2, 8] -> [2, 1]
)

# Lazy parameter initialization
Y = net(X)

print(net)
print(Y.shape)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=8, bias=True)
  (5): ReLU()
  (6): Linear(in_features=8, out_features=1, bias=True)
)
torch.Size([2, 1])


In [16]:
# Shared Parameter의 identity 확인

layer_2 = net[2]  # [8, 8]
layer_4 = net[4]  # [8, 8]

same_before = layer_2.weight.detach()[0] == layer_4.weight.detach()[0]

print(same_before)
print(net[2].weight is net[4].weight)

# 하나 임의로 수정해보기
with torch.no_grad():
    net[2].weight[0, 0] = 100

same_after = net[2].weight.detach()[0] == net[4].weight.detach()[0]

# 같은 object를 공유하고 있음을 확인
print(same_after)
print(net[4].weight[0, 0])

tensor([True, True, True, True, True, True, True, True])
True
tensor([True, True, True, True, True, True, True, True])
tensor(100., grad_fn=<SelectBackward0>)
